# 08_Finetune_affinity_module

### Create input for finetuning

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
seed=26

In [2]:
BitterDB_TP_TN_data = pd.read_excel('../data/BitterDB_TP_TN_data.xlsx')
BitterDB_TP_TN_data['complex'] = BitterDB_TP_TN_data['cID'].astype('str') + '-' + BitterDB_TP_TN_data['rID'].astype('str')
len(BitterDB_TP_TN_data)

8486

In [3]:
iptm_data = pd.read_excel('../data/BitterDB_TP_TN_data_iptm.xlsx')
reliable_complexes = iptm_data[iptm_data['reliable prediction']]['complex'].unique()
BitterDB_TP_TN_data = BitterDB_TP_TN_data[BitterDB_TP_TN_data['complex'].isin(reliable_complexes)].reset_index(drop=True)

In [4]:
BitterDB_TP_TN_data['association'].value_counts()

association
negative    6449
positive    1514
Name: count, dtype: int64

In [5]:
BitterDB_TP_TN_data['EC50, μM'].notna().value_counts()

EC50, μM
False    7519
True      444
Name: count, dtype: int64

In [6]:
def train_validation_test_split(df, validation_size, test_size, random_state, shuffle):
    train_validation_df, test_df = train_test_split(df, test_size=test_size, random_state=random_state, shuffle=shuffle)
    relative_validation_size = validation_size / (1 - test_size)
    train_df, validation_df = train_test_split(train_validation_df, test_size=relative_validation_size, random_state=random_state, shuffle=shuffle)
    return train_df, validation_df, test_df

In [7]:
train_dataset, validation_dataset, test_dataset = [], [], []
for (rid, association), sub_df in BitterDB_TP_TN_data.groupby(by=['rID', 'association']):
    if association == 'positive':

        ec50_values_df = sub_df.dropna(subset='EC50, μM', ignore_index=True)
        if not ec50_values_df.empty:
            if len(ec50_values_df) >= 3:
                train_subset, validation_subset, test_subset = train_validation_test_split(ec50_values_df, validation_size=0.1, test_size=0.1, random_state=seed, shuffle=True)
                train_dataset.append(train_subset)
                validation_dataset.append(validation_subset)
                test_dataset.append(test_subset)
            else:
                train_dataset.append(ec50_values_df)
        
        non_ec50_values_df = sub_df[sub_df['EC50, μM'].isna()].reset_index(drop=True)
        if not non_ec50_values_df.empty:
            if len(non_ec50_values_df) >= 3:
                train_subset, validation_subset, test_subset = train_validation_test_split(non_ec50_values_df, validation_size=0.1, test_size=0.1, random_state=seed, shuffle=True)
                train_dataset.append(train_subset)
                validation_dataset.append(validation_subset)
                test_dataset.append(test_subset)
            else:
                train_dataset.append(non_ec50_values_df)

    else:
        if len(sub_df) >= 3:
            train_subset, validation_subset, test_subset = train_validation_test_split(sub_df, validation_size=0.1, test_size=0.1, random_state=seed, shuffle=True)
            train_dataset.append(train_subset)
            validation_dataset.append(validation_subset)
            test_dataset.append(test_subset)
        else:
            train_dataset.append(sub_df)

In [8]:
train_dataset_df = pd.concat(train_dataset, ignore_index=True)
train_complexes = train_dataset_df['complex'].to_list()

validation_dataset_df = pd.concat(validation_dataset, ignore_index=True)
validation_complexes = validation_dataset_df['complex'].to_list()

test_dataset_df = pd.concat(test_dataset, ignore_index=True)
test_complexes = test_dataset_df['complex'].to_list()


In [9]:
finetune_labels = BitterDB_TP_TN_data[['complex', 'association', 'EC50, μM']].rename(columns={'complex': 'record_id', 'association': 'binary_label', 'EC50, μM': 'ec50'})
finetune_labels['binary_label'] = finetune_labels['binary_label'].map({'positive': 1, 'negative': 0})
finetune_labels['dataset'] = finetune_labels['record_id'].apply(lambda record_id: 'train' if record_id in train_complexes else ('validation' if record_id in validation_complexes else 'test'))
finetune_labels.to_csv('../predictions/Finetune_affinity_module/finetune_labels.csv', index=False)
finetune_labels['dataset'].value_counts()

dataset
train         6244
test           860
validation     859
Name: count, dtype: int64

### affinity fine-tuning

In [10]:
import torch
from torch.nn.utils import clip_grad_norm_
from pathlib import Path
import random
import numpy as np
from dataclasses import asdict
from typing import Iterable, Optional
from tqdm import tqdm

import sys
sys.path.append('../src/TAS2Rs_cofolding_utils/')
from finetune_data_preparation import prepare_splits
from finetune_dataset import AffinityFineTuneDataModule
from boltz.model.models.boltz2 import Boltz2
from boltz.data.module.inferencev2 import collate as collate_inference

from boltz.main import (
    Boltz2DiffusionParams,
    BoltzSteeringParams,
    PairformerArgsV2,
    MSAModuleArgs,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

[16:04:52] Initializing Normalizer


Using device: cuda


In [11]:
seed: int | None = 42
deterministic: bool = False

if seed is not None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
if deterministic:
    torch.use_deterministic_algorithms(True)

In [12]:
path_to_predictions = Path('../predictions/Cofolded_BitterDB_TP_TN/')
path_to_labels = Path('../predictions/Finetune_affinity_module/finetune_labels.csv')
path_to_checkpoint = Path('~/.boltz/boltz2_aff.ckpt')

splits = prepare_splits(
    path_to_predictions=path_to_predictions,
    path_to_labels=path_to_labels,
    iptm_threshold=False
)
print(f'Train manifest: {splits.path_to_train_manifest}')
print(f'Validation manifest: {splits.path_to_validation_manifest}')
print(f'Test manifest: {splits.path_to_test_manifest}')

print(f'Labels table: {splits.path_to_labels}')
if splits.dropped:
    print(f'Dropped {len(splits.dropped)} records missing artifacts or labels')

Number of predicted records: 8479
Number of available affinity labels: 7963
Number of records with missing prediction artifacts: 0
Number of dropped records: 516
Train manifest: ../predictions/Cofolded_BitterDB_TP_TN/processed/train_manifest.json
Validation manifest: ../predictions/Cofolded_BitterDB_TP_TN/processed/validation_manifest.json
Test manifest: ../predictions/Cofolded_BitterDB_TP_TN/processed/test_manifest.json
Labels table: ../predictions/Cofolded_BitterDB_TP_TN/processed/affinity_labels.csv
Dropped 516 records missing artifacts or labels


In [13]:
# Data modules
path_to_train_manifest = Path('../predictions/Cofolded_BitterDB_TP_TN/processed/train_manifest.json')
path_to_validation_manifest = Path('../predictions/Cofolded_BitterDB_TP_TN/processed/validation_manifest.json')
path_to_test_manifest = Path('../predictions/Cofolded_BitterDB_TP_TN/processed/test_manifest.json')
path_to_labels = Path('../predictions/Cofolded_BitterDB_TP_TN/processed/affinity_labels.csv')

data_module = AffinityFineTuneDataModule(
    path_to_train_manifest=path_to_train_manifest,
    path_to_validation_manifest=path_to_validation_manifest,
    path_to_test_manifest=path_to_test_manifest,
    path_to_labels=path_to_labels,
    path_to_predictions=path_to_predictions,
    batch_size=1,
    num_workers=16,
)
train_loader = data_module.train_dataloader()
val_loader = data_module.val_dataloader()
test_loader = data_module.test_dataloader()
print(f'Train batches: {len(train_loader)}, Validation batches: {len(val_loader)}, Test batches: {len(test_loader)}')

Train batches: 6244, Validation batches: 859, Test batches: 860


In [14]:
pairformer_args = PairformerArgsV2()
diffusion_params = Boltz2DiffusionParams()
msa_args = MSAModuleArgs()
steering_args = BoltzSteeringParams()
affinity_mw_correction: Optional[bool] = True
diffusion_samples_affinity: int = 3
sampling_steps_affinity: int = 200
predict_affinity_args = {
            "recycling_steps": 5,
            "sampling_steps": sampling_steps_affinity,
            "diffusion_samples": diffusion_samples_affinity,
            "max_parallel_samples": 1,
            "write_confidence_summary": False,
            "write_full_pae": False,
            "write_full_pde": False,
        }

# Load the base Boltz-2 checkpoint and focus gradients on affinity parameters only
model = Boltz2.load_from_checkpoint(path_to_checkpoint,
    strict=False,
    predict_args=predict_affinity_args,
    map_location=device,
    diffusion_process_args=asdict(diffusion_params),
    ema=False,
    pairformer_args=asdict(pairformer_args),
    msa_args=asdict(msa_args),
    steering_args=asdict(steering_args),
    affinity_mw_correction=affinity_mw_correction,
    )

model.structure_prediction_training = False
model.confidence_prediction = False
model.validate_structure = False
model.affinity_prediction = True
model.run_trunk_and_structure = False
model.skip_run_structure = True
model.affinity_prediction = True
model.use_kernels = False


for name, param in model.named_parameters():
    param.requires_grad = 'affinity_module2' in name # affinity
print('Trainable layers:', sum(p.requires_grad for p in model.parameters()))
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

/sci/labs/mashaniv/evgenii.ziaikin/anaconda3/envs/boltz-2/lib/python3.13/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0


Trainable layers: 185
Trainable parameters: 3251204


In [15]:
# Optimizer and loss definitions
import torch.nn.functional as F

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
pos_weight = torch.tensor([(1 - 0.18) / 0.18], device=device) ###
print(f'Pos_weight: {pos_weight}')
bce = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def move_batch(batch):
    moved = {}
    for key, value in batch.items():
        moved[key] = value.to(device) if torch.is_tensor(value) else value
    return moved

def compute_losses(preds, targets):
    binary_target = targets['affinity_binary_label'].float()
    logits = preds['affinity_logits_binary2']
    
    assert logits.shape == binary_target.shape, (logits.shape, binary_target.shape)
    binary_loss = bce(logits, binary_target)
    return binary_loss

Pos_weight: tensor([4.5556], device='cuda:0')


In [16]:
import torch.utils.checkpoint as cp
_orig = cp.checkpoint

def checkpoint_strip_use_trifast(fn, *args, **kwargs):
    kwargs.pop("use_trifast", None)
    return _orig(fn, *args, **kwargs)

cp.checkpoint = checkpoint_strip_use_trifast

In [17]:
import os
import math
from dataclasses import dataclass
from typing import Dict, Any, Optional, Tuple, List

import torch
import pandas as pd

In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, balanced_accuracy_score, f1_score, mean_absolute_error, root_mean_squared_error, r2_score

class AffinityPredictions:
    predictions: List[dict] = []

    def update_from_preds(self, preds: dict[str: torch.Tensor], targets: dict[str: torch.Tensor], record_id: str, stage: str, epoch: int):
        update_dict = {'record_id': record_id, 'stage': stage, 'epoch': epoch}
        preds.update(targets)
        for k, v in preds.items():
            update_dict[k] = v.item()
        self.predictions.append(update_dict)
    
    def _calculate_classification_metrics(self, df):
        Y_true = df['affinity_binary_label'].to_numpy(dtype=float)
        Y_pred = df['affinity_probability_binary2'].to_numpy(dtype=float).round()

        out = {
            'accuracy': accuracy_score(Y_true, Y_pred),
            'precision': precision_score(Y_true, Y_pred),
            'recall': recall_score(Y_true, Y_pred),
            'balanced_accuracy': balanced_accuracy_score(Y_true, Y_pred),
            'f1-score': f1_score(Y_true, Y_pred)
        }
        return pd.Series(out).round(2)
    
    def epoch_metrics(self, epoch):
        df = pd.DataFrame(self.predictions)
        df = df[df['epoch'] == epoch].reset_index(drop=True)

        epoch_metrics = {}

        for stage, sub_df in df.groupby(by='stage'):
            epoch_metrics[stage] = self._calculate_classification_metrics(sub_df)
        
        return epoch_metrics
    
    def save_predictions(self, path_to_file):
        df = pd.DataFrame(self.predictions)
        df.to_excel(path_to_file, index=False)

In [19]:
class AffinityLosses:

    losses: List[dict] = []

    def __init__(self, accum_steps: int):
        self.accum_steps = accum_steps

    def update_losses(self, b_loss: torch.Tensor, record_id: str, stage: str, step: int, epoch: int):
        update_dict = {'epoch': epoch, 'stage': stage, 'step': step, 'record_id': record_id, 'b_loss': b_loss.item() / self.accum_steps}
        self.losses.append(update_dict)
    
    def epoch_losses(self, epoch):
        df = pd.DataFrame(self.losses)
        df = df[df['epoch'] == epoch].reset_index(drop=True)

        epoch_losses = {}

        for stage, sub_df in df.groupby(by='stage'):
            b_loss = sub_df['b_loss'].sum() / sub_df['step'].max()
            epoch_losses[stage] = b_loss.round(4)
        
        return epoch_losses
    
    def save_losses(self, path_to_file):
        df = pd.DataFrame(self.losses)
        df.to_excel(path_to_file, index=False)

In [20]:
# Training loop
num_epochs: int = 20
accum_steps = 32

epoch_affinity_predictions = AffinityPredictions()
epoch_affinity_losses = AffinityLosses(accum_steps)

In [21]:
epoch = 0

stage = 'train'
print(f'{stage} step')

model.eval()
with torch.no_grad():
    for n, batch in enumerate(tqdm(train_loader), start=1):
        record_id = batch['record'][0].id

        batch = move_batch(batch)
        targets = {k: batch.pop(k) for k in ['affinity_binary_label']}
        
        out = model(batch)
        preds = {k: out.pop(k) for k in ['affinity_probability_binary2', 'affinity_logits_binary2']}

        epoch_affinity_predictions.update_from_preds(preds, targets, record_id, stage, epoch)

        b_loss = compute_losses(preds, targets)

        step = (n - 1) // accum_steps + 1
        epoch_affinity_losses.update_losses(b_loss, record_id, stage, step, epoch)

stage = 'validation'
print(f'{stage} step')

model.eval()
with torch.no_grad():
    for n, batch in enumerate(tqdm(val_loader), start=1):
        record_id = batch['record'][0].id

        batch = move_batch(batch)
        targets = {k: batch.pop(k) for k in ['affinity_binary_label']}

        out = model(batch)
        preds = {k: out.pop(k) for k in ['affinity_probability_binary2', 'affinity_logits_binary2']}

        epoch_affinity_predictions.update_from_preds(preds, targets, record_id, stage, epoch)
        
        b_loss = compute_losses(preds, targets)

        step = (n - 1) // accum_steps + 1
        epoch_affinity_losses.update_losses(b_loss, record_id, stage, step, epoch)

stage = 'test'
print(f'{stage} step')

model.eval()
with torch.no_grad():
    for n, batch in enumerate(tqdm(test_loader), start=1):
        record_id = batch['record'][0].id

        batch = move_batch(batch)
        targets = {k: batch.pop(k) for k in ['affinity_binary_label']}

        out = model(batch)
        preds = {k: out.pop(k) for k in ['affinity_probability_binary2', 'affinity_logits_binary2']}

        epoch_affinity_predictions.update_from_preds(preds, targets, record_id, stage, epoch)
        
        b_loss = compute_losses(preds, targets)

        step = (n - 1) // accum_steps + 1
        epoch_affinity_losses.update_losses(b_loss, record_id, stage, step, epoch)

print(f'Epoch {epoch}: \n')
print('losses')
for stage, losses in epoch_affinity_losses.epoch_losses(epoch).items():
    print(stage)
    print(losses)
print('\n')

print('metrcis')
for stage, predictions in epoch_affinity_predictions.epoch_metrics(epoch).items():
    print(stage)
    print(predictions)
print('\n')

epoch_affinity_predictions.save_predictions('../predictions/Finetune_affinity_module/epoch_predictions.xlsx')
epoch_affinity_losses.save_losses('../predictions/Finetune_affinity_module/epoch_losses.xlsx')

torch.cuda.empty_cache()

train step


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6244/6244 [17:37<00:00,  5.90it/s]


validation step


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 859/859 [02:23<00:00,  5.99it/s]


test step


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 860/860 [02:24<00:00,  5.97it/s]


Epoch 0: 

losses
test
2.3056
train
2.3863
validation
2.3144


metrcis
test
accuracy             0.23
precision            0.22
recall               0.99
balanced_accuracy    0.50
f1-score             0.36
dtype: float64
train
accuracy             0.19
precision            0.18
recall               1.00
balanced_accuracy    0.50
f1-score             0.31
dtype: float64
validation
accuracy             0.23
precision            0.22
recall               1.00
balanced_accuracy    0.51
f1-score             0.36
dtype: float64




In [22]:
import pytorch_lightning as pl

def save_boltz_checkpoint(model: Boltz2, checkpoint_path: Path | str) -> Path:
    """Save a Boltz2 model to a Lightning checkpoint for future loading."""
    checkpoint = {
        "state_dict": model.state_dict(),
        "hyper_parameters": dict(model.hparams),
        "pytorch-lightning_version": pl.__version__,
        "epoch": 0,
        "global_step": 0,
    }
    model.on_save_checkpoint(checkpoint)

    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint, checkpoint_path)
    return checkpoint_path

In [ ]:
for epoch in range(1, num_epochs + 1):

    stage = 'train'
    print(f'{stage} step')

    model.train()
    optimizer.zero_grad(set_to_none=True)
    for n, batch in enumerate(tqdm(train_loader), start=1):
        record_id = batch['record'][0].id

        batch = move_batch(batch)
        targets = {k: batch.pop(k) for k in ['affinity_binary_label']}
        
        out = model(batch)
        preds = {k: out.pop(k) for k in ['affinity_probability_binary2', 'affinity_logits_binary2']}
        epoch_affinity_predictions.update_from_preds(preds, targets, record_id, stage, epoch)

        b_loss = compute_losses(preds, targets)
        (b_loss / accum_steps).backward()

        step = (n - 1) // accum_steps + 1
        epoch_affinity_losses.update_losses(b_loss, record_id, stage, step, epoch)

        if n % accum_steps == 0:
            clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
    
    if n % accum_steps != 0:
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
    
    stage = 'validation'
    print(f'{stage} step')

    model.eval()
    with torch.no_grad():
        for n, batch in enumerate(tqdm(val_loader), start=1):
            record_id = batch['record'][0].id

            batch = move_batch(batch)
            targets = {k: batch.pop(k) for k in ['affinity_binary_label']}

            out = model(batch)
            preds = {k: out.pop(k) for k in ['affinity_probability_binary2', 'affinity_logits_binary2']}

            epoch_affinity_predictions.update_from_preds(preds, targets, record_id, stage, epoch)
            
            b_loss = compute_losses(preds, targets)

            step = (n - 1) // accum_steps + 1
            epoch_affinity_losses.update_losses(b_loss, record_id, stage, step, epoch)

    stage = 'test'
    print(f'{stage} step')
    
    model.eval()
    with torch.no_grad():
        for n, batch in enumerate(tqdm(test_loader), start=1):
            record_id = batch['record'][0].id
    
            batch = move_batch(batch)
            targets = {k: batch.pop(k) for k in ['affinity_binary_label']}
    
            out = model(batch)
            preds = {k: out.pop(k) for k in ['affinity_probability_binary2', 'affinity_logits_binary2']}
    
            epoch_affinity_predictions.update_from_preds(preds, targets, record_id, stage, epoch)
            
            b_loss = compute_losses(preds, targets)
    
            step = (n - 1) // accum_steps + 1
            epoch_affinity_losses.update_losses(b_loss, record_id, stage, step, epoch)

    print(f'Epoch {epoch}: \n')
    print('losses')
    for stage, losses in epoch_affinity_losses.epoch_losses(epoch).items():
        print(stage)
        print(losses)
    print('\n')
    
    print('metrcis')
    for stage, predictions in epoch_affinity_predictions.epoch_metrics(epoch).items():
        print(stage)
        print(predictions)
    print('\n')

    epoch_affinity_predictions.save_predictions('../predictions/Finetune_affinity_module/epoch_predictions.xlsx')
    epoch_affinity_losses.save_losses('../predictions/Finetune_affinity_module/epoch_losses.xlsx')
    
    checkpoint_path = f'../predictions/Finetune_affinity_module/checkpoints/boltz2_aff_TAS2Rs_epoch_{epoch}.ckpt'
    save_boltz_checkpoint(model, checkpoint_path)

    torch.cuda.empty_cache()

train step


  0%|                                                                                                                           | 0/6244 [00:00<?, ?it/s]/sci/labs/mashaniv/evgenii.ziaikin/anaconda3/envs/boltz-2/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/sci/labs/mashaniv/evgenii.ziaikin/anaconda3/envs/boltz-2/lib/python3.13/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████

validation step


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 859/859 [04:19<00:00,  3.31it/s]


test step


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 860/860 [03:08<00:00,  4.56it/s]


... (4 outputs left out)

test step


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 860/860 [05:05<00:00,  2.82it/s]


Epoch 2: 

losses
test
1.1953
train
1.1128
validation
1.1563


metrcis
test
accuracy             0.58
precision            0.30
recall               0.68
balanced_accuracy    0.61
f1-score             0.42
dtype: float64
train
accuracy             0.60
precision            0.25
recall               0.60
balanced_accuracy    0.60
f1-score             0.35
dtype: float64
validation
accuracy             0.63
precision            0.33
recall               0.71
balanced_accuracy    0.66
f1-score             0.45
dtype: float64


train step


  0%|                                                                                                                           | 0/6244 [00:00<?, ?it/s]/sci/labs/mashaniv/evgenii.ziaikin/anaconda3/envs/boltz-2/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/sci/labs/mashaniv/evgenii.ziaikin/anaconda3/envs/boltz-2/lib/python3.13/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
  2%|██▍                                                                                                         